[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/03_setup/03_setup_check.ipynb)


# 03 — 환경 구성 — 설치·검증·러닝 예제

> 본문 [`03_setup.md`](03_setup.md) 와 **한 절씩 짝지어** 보세요.
> **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행해요.
> 이 노트북의 표·그래프·수치는 **여러분이 방금 돌린 `my_run/`** 에서 나와요. 실행을 건너뛰거나 실패하면 저장소에 커밋된 `data/`(실제 실행 산출물) 로 자동 폴백해요.
>
> **실측 소요 —** ANARCI numbering+germline **0.15초** · Sapiens 첫 실행(모델 가중치 다운로드 포함) **6.3초**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`03_setup`) 폴더로 이동한 뒤, 필요한 패키지만 깝니다.
- **로컬**: 이 노트북을 `03_setup/` 폴더 안에서 열었다면 클론 없이 그대로 진행돼요.

이 셀이 끝나면 두 개의 경로가 준비돼요 — **`my_run/`**(내가 직접 만들 결과)과 **`data/`**(저장소에 커밋된 레퍼런스 결과).
아래 랩은 항상 `my_run/` 을 먼저 찾고, 없으면 `data/` 로 폴백하면서 **어느 쪽을 쓰는지 print** 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌려요. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행돼요 — pip 만으로는 `hmmscan` 이 없어 죽어요.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "03_setup"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

# ANARCI 는 numbering 을 hmmscan(HMMER) 서브프로세스로 돌려요 — pip 만으로는 hmmscan 이 없어요.
if APT_PKGS and IN_COLAB:
    _run("apt-get -qq update")                 # 인덱스가 낡으면 install 이 404 로 죽는다
    _run(f"apt-get -qq install -y {APT_PKGS}")
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨져요.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — 도구 설치 (본문 3.1~3.2)

Colab 기준 **한 줄씩** 깝니다. 순서가 중요해요.

```bash
apt-get -qq update                    # ① apt 인덱스 갱신
apt-get -qq install -y hmmer          # ② ANARCI 가 부르는 hmmscan 바이너리
pip -q install anarci abnumber        # ② numbering (pip 만으로는 hmmscan 이 없어 실패)
pip -q install sapiens                # ③ humanization 엔진 (Ch.05)
```

로컬 conda 라면 본문 3.1 의 `conda create -n abhuman -c conda-forge -c bioconda python=3.10 anarci hmmer` 로 대체할 수 있어요.
어느 경로든 **`hmmscan` 이 PATH 에 있어야** 합니다 — 이 한 줄이 Ch.04·06·07 에서 가장 흔한 에러를 막아요.

In [ ]:
t0 = time.time()
if IN_COLAB:
    _run("apt-get -qq update")
    _run("apt-get -qq install -y hmmer")
_ensure("anarci abnumber sapiens")
print(f"\n설치 셀 소요: {time.time()-t0:.1f}초")

## 2) 내 결과 확인 — 환경 진단 (본문 3.4)

`import` 가 되는지, 그리고 **`hmmscan` 이 실제로 PATH 에 있는지**를 함께 봅니다.
CLI `ANARCI` 가 돌아가도 파이썬 모듈 `anarci` 가 import 되지 않으면 Humatch·AbNatiV 가 뒤에서 죽어요.

In [ ]:
import importlib, shutil, pandas as pd

def probe(mod):
    try:
        m = importlib.import_module(mod)
        return "OK", getattr(m, "__version__", "?")
    except Exception as e:
        return "FAIL", f"{type(e).__name__}: {e}"[:60]

rows = []
for mod in ["anarci", "abnumber", "sapiens", "pandas"]:
    st, ver = probe(mod)
    rows.append({"항목": f"import {mod}", "상태": st, "버전/메시지": ver})
rows.append({"항목": "hmmscan (HMMER)", "상태": "OK" if shutil.which("hmmscan") else "FAIL",
             "버전/메시지": shutil.which("hmmscan") or "PATH 에 없음 → ANARCI numbering 이 FileNotFoundError 로 죽어요"})
rows.append({"항목": "ANARCI CLI", "상태": "OK" if shutil.which("ANARCI") else "FAIL",
             "버전/메시지": shutil.which("ANARCI") or "없음(파이썬 API 로도 진행 가능)"})

env = pd.DataFrame(rows)
MY.mkdir(exist_ok=True)
env.to_csv(MY / "env_report.csv", index=False)
print("내 환경 리포트 →", MY / "env_report.csv")
env

## 3) 직접 실행 — 러닝 예제 서열로 첫 numbering

가이드 전체를 관통하는 예제 서열(`data/parental.fasta`)입니다. Ch.04~10 의 모든 수치가 이 두 체인에서 나와요.
여기서는 도구가 실제로 도는지만 확인해요 — 본격적인 numbering·CDR 추출은 Ch.04 에서 합니다.

In [ ]:
try:
    from abnumber import Chain

    t0 = time.time()
    ch_h = Chain(VH, scheme="imgt")
    ch_l = Chain(VL, scheme="imgt")
    el = time.time() - t0

    print(f"VH  : {len(VH)} aa | chain_type={ch_h.chain_type} | CDR3={ch_h.cdr3_seq}")
    print(f"VL  : {len(VL)} aa | chain_type={ch_l.chain_type} | CDR3={ch_l.cdr3_seq}")
    print(f"\nabnumber numbering 소요: {el:.2f}초")

    smoke = {"VH_len": len(VH), "VL_len": len(VL),
             "VH_cdr3": ch_h.cdr3_seq, "VL_cdr3": ch_l.cdr3_seq,
             "VH_chain_type": ch_h.chain_type, "VL_chain_type": ch_l.chain_type}
    (MY / "smoke.json").write_text(json.dumps(smoke, ensure_ascii=False, indent=1))
    print("→", MY / "smoke.json")
except Exception as e:
    print("스모크 실패:", type(e).__name__, str(e)[:160])
    print("→ 2) 진단표에서 FAIL 인 항목을 보고 1) 설치 셀을 다시 실행하세요.")
    print("   (hmmscan 이 없으면 numbering 이, abnumber 가 없으면 CDR 추출이 여기서 막혀요.)")

## 4) 레퍼런스 대조 — 이 가이드를 검증한 도구 버전

`data/verified_versions.csv` 는 이 가이드의 모든 수치를 뽑을 때 **실제로 쓴 버전**입니다.
내 환경의 버전이 달라도 대개 문제없지만, 결과가 어긋나면 여기부터 비교하세요.

In [ ]:
ver = pd.read_csv("data/verified_versions.csv")
display(ver)

mine = {m: probe(m)[1] for m in ["anarci", "abnumber", "sapiens"]}
print("\n내 환경 버전:", mine)
print("\n주의 — torch 2.13.0+cpu 휠은 import 시 undefined symbol 로 깨져요(실측). 2.6.0 을 쓰세요.")

## 이 랩에서 확인한 것

1. `hmmscan`(HMMER) 이 PATH 에 없으면 ANARCI numbering 이 **FileNotFoundError** 로 죽어요 — Colab 은 `apt-get install -y hmmer` 가 정답.
2. 설치 채널이 도구마다 달라요 — Colab/PyPI 경로는 `anarci`·`abnumber`·`sapiens`·`anthroab`·`abnativ` 가 pip, Humatch 는 pip 이 안 되면 GitHub source(Ch.06 에서 자동 처리).
3. 러닝 예제 CDR3 — VH `ARRGRYGLYAMDY` · VL `QSYDSSLRVV`. 이 값이 나왔다면 Ch.04 로 넘어가도 좋아요.

다음 → **[Ch.04 — 입력 QC](../04_sequence_qc/04_numbering_lab.ipynb)**